In [1]:
import geopandas as gpd

In [2]:
address_gdf = gpd.read_parquet("../../data/processed/nsw_addressing/addresspoint_all.parquet")
heritage_gdf = gpd.read_parquet("../../data/processed/nsw_heritage/heritage.parquet")

In [3]:
print("Address rows:", len(address_gdf))
print("Heritage rows:", len(heritage_gdf))

print("Address CRS:", address_gdf.crs)
print("Heritage CRS:", heritage_gdf.crs)

print(address_gdf.columns)
print(heritage_gdf.columns)

Address rows: 4225113
Heritage rows: 40422
Address CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direc

In [5]:
heritage_keep = heritage_gdf[
    [
        "OBJECTID",
        "EPI_NAME",
        "LGA_NAME",
        "LAY_CLASS",
        "H_ID",
        "H_NAME",
        "SIG",
        "EPI_TYPE",
        "geometry",
    ]
].copy()

if address_gdf.crs != heritage_keep.crs:
    address_gdf = address_gdf.to_crs(heritage_keep.crs)

sample_address = address_gdf.sample(50000, random_state=42).copy()
print(len(sample_address))

50000


In [6]:
joined = gpd.sjoin(
    sample_address,
    heritage_keep,
    how="left",
    predicate="within"
)

print("Joined rows:", len(joined))
joined[
    ["address", "LAY_CLASS", "H_NAME", "SIG", "EPI_NAME", "LGA_NAME"]
].head(10)

Joined rows: 50469


,address,LAY_CLASS,H_NAME,SIG,EPI_NAME,LGA_NAME
2262872,GRINTON AVENUE ASHMONT,NaN,NaN,NaN,NaN,NaN
136524,58 VISTA AVENUE CATALINA,NaN,NaN,NaN,NaN,NaN
3400623,5/40 TRAIN STREET BROULEE,NaN,NaN,NaN,NaN,NaN
975625,13 BACH AVENUE EMERTON,NaN,NaN,NaN,NaN,NaN
1229084,12 GUILFOYLE PLACE CUDGEN,NaN,NaN,NaN,NaN,NaN
3506896,14 SPURFIELD ROAD MCLEANS RIDGES,NaN,NaN,NaN,NaN,NaN
2094798,1/43 THE TRONGATE GRANVILLE,NaN,NaN,NaN,NaN,NaN
1372500,39 MANNS ROAD NARARA,NaN,NaN,NaN,NaN,NaN
466580,4 CAROLINA CRESCENT MUDGEE,NaN,NaN,NaN,NaN,NaN
3991008,62 WADE STREET COOLAMON,NaN,NaN,NaN,NaN,NaN


In [7]:
joined["LAY_CLASS"].isna().mean()

np.float64(0.9208226832312905)

In [8]:
joined["LAY_CLASS"].value_counts(dropna=False).head(20)

LAY_CLASS
NaN                                          46473
Conservation Area - General                   2850
Item - General                                1034
Item - Archaeological                           53
Conservation Area - Landscape                   22
Item - Landscape                                16
Conservation Area - Aboriginal                  14
Local Heritage - General                         3
Aboriginal Place of Heritage Significance        3
Conservation Area - Archaeological               1
Name: count, dtype: int64

In [13]:
joined["rid"].duplicated().mean()

np.float64(0.009292833224355545)

In [9]:
heritage_hits = joined[joined["LAY_CLASS"].notna()].copy()
print("Heritage hits:", len(heritage_hits))

heritage_hits[
    ["address", "LAY_CLASS", "H_NAME", "SIG", "EPI_NAME", "LGA_NAME"]
].sample(min(20, len(heritage_hits)), random_state=42)

Heritage hits: 3996


,address,LAY_CLASS,H_NAME,SIG,EPI_NAME,LGA_NAME
930579,48-50 PARK ROAD BURWOOD,Item - General,"""Rossmoyne"" - Formerly ""Tulloona""",Local,Burwood Local Environmental Plan 2012,BURWOOD
541,25 THOMPSON STREET DRUMMOYNE,Conservation Area - General,Bourketown,Local,Canada Bay Local Environmental Plan 2013,CANADA BAY
85146,207 TRAFALGAR STREET ANNANDALE,Conservation Area - General,Annandale Heritage Conservation Area,Local,Inner West Local Environmental Plan 2022,INNER WEST
3301947,76 CHALLONER RISE RENWICK,Item - General,"Former Renwick Institution, including brick si...",Local,Wingecarribee Local Environmental Plan 2010,WINGECARRIBEE
1260416,49 ELIZABETH STREET ARTARMON,Conservation Area - General,Artamon,Local,Willoughby Local Environmental Plan 2012,WILLOUGHBY
1534901,58 CHAPMAN AVENUE BEECROFT,Conservation Area - General,"Beecroft, Cheltenham Heritage Conservation Area",Local,Hornsby Local Environmental Plan 2013,HORNSBY
2640296,33 CHELMSFORD PLACE LEETON,Item - General,Leeton District Lands Office,State,Leeton Local Environmental Plan 2014,LEETON
2866482,17 TENTH AVENUE WARILLA,Item - Landscape,Lake Windermere Caravan Park pine trees,Local,Shellharbour Local Environmental Plan 2013,SHELLHARBOUR
3064692,33E/18 LUCY STREET ASHFIELD,Conservation Area - General,Lucy Street Heritage Conservation Area,Local,Inner West Local Environmental Plan 2022,INNER WEST
3067660,55/347 LIVERPOOL STREET DARLINGHURST,Item - General,Flat building 'Mont Clair',Local,Sydney Local Environmental Plan 2012,SYDNEY


In [10]:
joined["rid"].duplicated().mean()

np.float64(0.009292833224355545)

In [11]:
joined["heritage_flag"] = joined["LAY_CLASS"].notna().astype(int)
joined["heritage_flag"].value_counts(normalize=True)

heritage_flag
0    0.920823
1    0.079177
Name: proportion, dtype: float64

In [12]:
from pathlib import Path

Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

joined.to_parquet(
    "../../data/processed/site_features/address_with_heritage_sample.parquet",
    index=False
)